# Treccani validation of Cultura

## Settings

In [1]:
from pathlib import Path
import json
import re

import duckdb
import pandas as pd

ROOT = Path.cwd().parent
AI_ANSWERS = ROOT / "scripts" / "5-annotations" / "treccani_validation" / "treccani_ai_annotations_cultura289.parquet"
DATABASE = ROOT / "data" / "cultura" / "humans_clean_v3_sample_treccani.duckdb"
REFERENCES = ROOT / "scripts" / "5-annotations" / "data" / "treccani_wikidata_references.json"
TRECCANI_CHECKED = ["P569", "P570", "P27", "P20", "P19"]

## Treccani answers found by the AI, without individuals referenced to Treccani

In [2]:
def window_years(answer):
    answer = answer if isinstance(answer, str) else ""
    years = []
    for year, decade in re.findall(r"\b(\d{3,4})(s?)\b", answer):
        years += [int(year), int(year) + 9] if decade else [int(year)]
    if years:
        return min(years), max(years)
    centuries = [int(c) for c in re.findall(r"(\d{1,2})(?:st|nd|rd|th) century", answer)]
    if centuries:
        return (min(centuries) - 1) * 100 + 1, max(centuries) * 100
    return None, None


raw = pd.read_parquet(AI_ANSWERS)
treccani = pd.DataFrame({
    "qid": raw["entity"].map(lambda entity: entity["qid"]),
    "name": raw["entity"].map(lambda entity: entity["label_en"]),
    "treccani_polity": raw["polity_ai_matched"].map(lambda answer: answer["answer"]),
    "treccani_polity_id": raw["polity_ai_matched"].map(lambda answer: int(answer["source_verbatim"][0].split("|")[0]) if len(answer["source_verbatim"]) else None),
    "treccani_window": raw["productivity_window_ai_extracted"].map(lambda answer: answer["answer"]),
})
treccani[["treccani_start", "treccani_end"]] = treccani["treccani_window"].map(window_years).tolist()
assert treccani["qid"].is_unique
referenced_to_treccani = {s["qid"] for s in json.loads(REFERENCES.read_text()) if s["status"] == "treccani" and s["pid"] in TRECCANI_CHECKED}
print(len(treccani), "individuals annotated")
treccani = treccani[~treccani["qid"].isin(referenced_to_treccani)].reset_index(drop=True)
print(len(treccani), "kept after removing those with a birth date, death date, citizenship, birthplace or deathplace referenced to Treccani")
treccani

289 individuals annotated
178 kept after removing those with a birth date, death date, citizenship, birthplace or deathplace referenced to Treccani


,qid,name,treccani_polity,treccani_polity_id,treccani_window,treccani_start,treccani_end
0,Q103928601,Antonio Martinelli,Republic of Venice,397.0,from the mid-1720s,1720.0,1729.0
1,Q1056851,Cesare I Gonzaga,Holy Roman Empire Minor States,783.0,1557-1575,1557.0,1575.0
2,Q107557613,Francesco Bembo,Republic of Venice,397.0,1333-1378,1333.0,1378.0
3,Q1092397,Angelo Nicolini,NaN,NaN,1523-1567,1523.0,1567.0
4,Q109838240,Paolo Catania,Spanish Empire,1622.0,1610-1665,1610.0,1665.0
...,...,...,...,...,...,...,...
173,Q952234,Giovanni Paolo Lascaris,NaN,NaN,1636-1657,1636.0,1657.0
174,Q978985,Leo of Ostia,Norman Italy,608.0,1071-1115,1071.0,1115.0
175,Q982080,Domenico Montagnana,Republic of Venice,397.0,starting from 1715,1715.0,1715.0
176,Q98431860,Giovanni Battista de Cavalletto,Papal States,419.0,1486-1523,1486.0,1523.0


## Cultura window and polities

In [3]:
connection = duckdb.connect(str(DATABASE), read_only=True)
qids = treccani["qid"].tolist()

windows = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           individual_enriched.peak_productivity.start_year AS cultura_start,
           individual_enriched.peak_productivity.end_year AS cultura_end
    FROM individual_enriched
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
""", [qids]).df()
windows["cultura_window"] = windows.apply(lambda row: None if pd.isna(row["cultura_start"]) else f"{int(row['cultura_start'])}-{int(row['cultura_end'])}", axis=1)

sub_polities = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           list(polity_match.polity.cliopatria_id) AS cultura_sub_polity_ids,
           string_agg(polity_match.polity.name, ' / ') AS cultura_sub_polities
    FROM individual_enriched, unnest(individual_enriched.polity) AS unnested(polity_match)
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
    GROUP BY qid
""", [qids]).df()

meta_polities = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           list(DISTINCT meta.cliopatria_id) AS cultura_meta_polity_ids,
           string_agg(DISTINCT meta.name, ' / ') AS cultura_meta_polities
    FROM individual_enriched,
         unnest(individual_enriched.polity) AS unnested(polity_match),
         unnest(polity_match.polity.meta_polities) AS unnested_meta(meta)
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
      AND meta.start_year <= individual_enriched.peak_productivity.end_year
      AND meta.end_year >= individual_enriched.peak_productivity.start_year
    GROUP BY qid
""", [qids]).df()

cultura = windows.merge(sub_polities, on="qid", how="left").merge(meta_polities, on="qid", how="left")
print(len(cultura), "individuals,", cultura["cultura_sub_polities"].notna().sum(), "with a polity")
cultura

178 individuals, 177 with a polity


,qid,cultura_start,cultura_end,cultura_window,cultura_sub_polity_ids,cultura_sub_polities,cultura_meta_polity_ids,cultura_meta_polities
0,Q3638721,1514,1538,1514-1538,"[1000, 699]",Kingdom of Spain / Crown of Aragon,"[975, 1622]",Personal union of Crown of Aragon with Crown o...
1,Q94640805,1876,1879,1876-1879,[350],Kingdom of Italy,NaN,NaN
2,Q1525438,1949,1979,1949-1979,[1428],Republic of Italy,NaN,NaN
3,Q132708,1643,1691,1643-1691,[419],Papal States,NaN,NaN
4,Q1374872,1485,1508,1485-1508,[397],Republic of Venice,NaN,NaN
...,...,...,...,...,...,...,...,...
173,Q3846252,1770,1787,1770-1787,[397],Republic of Venice,NaN,NaN
174,Q3108086,1684,1707,1684-1707,[419],Papal States,NaN,NaN
175,Q98960294,1557,1560,1557-1560,[397],Republic of Venice,NaN,NaN
176,Q3791437,1296,1296,1296-1296,[783],Holy Roman Empire Minor States,[1612],Holy Roman Empire


## Comparison table

In [4]:
comparison = treccani.merge(cultura, on="qid", how="left")
assert len(comparison) == len(treccani)
comparison

,qid,name,treccani_polity,treccani_polity_id,treccani_window,treccani_start,treccani_end,cultura_start,cultura_end,cultura_window,cultura_sub_polity_ids,cultura_sub_polities,cultura_meta_polity_ids,cultura_meta_polities
0,Q103928601,Antonio Martinelli,Republic of Venice,397.0,from the mid-1720s,1720.0,1729.0,1731,1754,1731-1754,[397],Republic of Venice,NaN,NaN
1,Q1056851,Cesare I Gonzaga,Holy Roman Empire Minor States,783.0,1557-1575,1557.0,1575.0,1567,1575,1567-1575,[783],Holy Roman Empire Minor States,[1612],Holy Roman Empire
2,Q107557613,Francesco Bembo,Republic of Venice,397.0,1333-1378,1333.0,1378.0,1301,1500,1301-1500,[397],Republic of Venice,NaN,NaN
3,Q1092397,Angelo Nicolini,NaN,NaN,1523-1567,1523.0,1567.0,1536,1551,1536-1551,[783],Holy Roman Empire Minor States,[1612],Holy Roman Empire
4,Q109838240,Paolo Catania,Spanish Empire,1622.0,1610-1665,1610.0,1665.0,1622,1643,1622-1643,"[1043, 1000]",Iberian Union / Kingdom of Spain,[1622],Spanish Empire
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,Q952234,Giovanni Paolo Lascaris,NaN,NaN,1636-1657,1636.0,1657.0,1587,1600,1587-1600,"[1034, 570]",House of Savoy / Kingdom of France,[1616],Kingdom of France
174,Q978985,Leo of Ostia,Norman Italy,608.0,1071-1115,1071.0,1115.0,1081,1091,1081-1091,[419],Papal States,NaN,NaN
175,Q982080,Domenico Montagnana,Republic of Venice,397.0,starting from 1715,1715.0,1715.0,1712,1740,1712-1740,[397],Republic of Venice,NaN,NaN
176,Q98431860,Giovanni Battista de Cavalletto,Papal States,419.0,1486-1523,1486.0,1523.0,1514,1523,1514-1523,[419],Papal States,NaN,NaN


## Polity accuracy

In [5]:
as_list = lambda ids: [] if ids is None or isinstance(ids, float) else list(ids)
names = lambda text: [] if not isinstance(text, str) else [name.strip() for name in text.split(" / ")]
sub_ids = comparison["cultura_sub_polity_ids"].map(as_list)
meta_ids = comparison["cultura_meta_polity_ids"].map(as_list)
sub_names = comparison["cultura_sub_polities"].map(names)
meta_names = comparison["cultura_meta_polities"].map(names)
comparison["polity_match"] = [
    "sub polity" if treccani_id in subs or treccani_name in sub_name_list else
    "meta polity" if treccani_id in metas or treccani_name in meta_name_list else "none"
    for treccani_id, treccani_name, subs, metas, sub_name_list, meta_name_list
    in zip(comparison["treccani_polity_id"], comparison["treccani_polity"], sub_ids, meta_ids, sub_names, meta_names)
]
comparison["polity_ok"] = comparison["polity_match"] != "none"
comparison["has_treccani_polity"] = comparison["treccani_polity_id"].notna()
print((~comparison["has_treccani_polity"]).sum(), "individual(s) without a Treccani polity, left out of the polity accuracy")
comparison[["name", "treccani_polity", "cultura_sub_polities", "cultura_meta_polities", "polity_match", "polity_ok"]]

8 individual(s) without a Treccani polity, left out of the polity accuracy


,name,treccani_polity,cultura_sub_polities,cultura_meta_polities,polity_match,polity_ok
0,Antonio Martinelli,Republic of Venice,Republic of Venice,NaN,sub polity,True
1,Cesare I Gonzaga,Holy Roman Empire Minor States,Holy Roman Empire Minor States,Holy Roman Empire,sub polity,True
2,Francesco Bembo,Republic of Venice,Republic of Venice,NaN,sub polity,True
3,Angelo Nicolini,NaN,Holy Roman Empire Minor States,Holy Roman Empire,none,False
4,Paolo Catania,Spanish Empire,Iberian Union / Kingdom of Spain,Spanish Empire,meta polity,True
...,...,...,...,...,...,...
173,Giovanni Paolo Lascaris,NaN,House of Savoy / Kingdom of France,Kingdom of France,none,False
174,Leo of Ostia,Norman Italy,Papal States,NaN,none,False
175,Domenico Montagnana,Republic of Venice,Republic of Venice,NaN,sub polity,True
176,Giovanni Battista de Cavalletto,Papal States,Papal States,NaN,sub polity,True


## Productivity window accuracy

In [6]:
overlap = (comparison[["treccani_end", "cultura_end"]].min(axis=1, skipna=False) - comparison[["treccani_start", "cultura_start"]].max(axis=1, skipna=False) + 1).clip(lower=0)
treccani_span = comparison["treccani_end"] - comparison["treccani_start"] + 1
comparison["overlap_years"] = overlap
comparison["overlap_share"] = (overlap / treccani_span).round(2)
comparison["window_ok"] = comparison["overlap_years"].fillna(0) > 0
comparison["has_treccani_window"] = comparison["treccani_start"].notna()
print((~comparison["has_treccani_window"]).sum(), "individual(s) without a Treccani window, left out of the window accuracy")
print(comparison["cultura_start"].isna().sum(), "individual(s) without a Cultura window, counted as wrong")
comparison[["name", "treccani_window", "cultura_window", "overlap_years", "overlap_share", "window_ok"]]

1 individual(s) without a Treccani window, left out of the window accuracy
1 individual(s) without a Cultura window, counted as wrong


,name,treccani_window,cultura_window,overlap_years,overlap_share,window_ok
0,Antonio Martinelli,from the mid-1720s,1731-1754,0.0,0.0,False
1,Cesare I Gonzaga,1557-1575,1567-1575,9.0,0.47,True
2,Francesco Bembo,1333-1378,1301-1500,46.0,1.0,True
3,Angelo Nicolini,1523-1567,1536-1551,16.0,0.36,True
4,Paolo Catania,1610-1665,1622-1643,22.0,0.39,True
...,...,...,...,...,...,...
173,Giovanni Paolo Lascaris,1636-1657,1587-1600,0.0,0.0,False
174,Leo of Ostia,1071-1115,1081-1091,11.0,0.24,True
175,Domenico Montagnana,starting from 1715,1712-1740,1.0,1.0,True
176,Giovanni Battista de Cavalletto,1486-1523,1514-1523,10.0,0.26,True


## Accuracy of Cultura against Treccani

In [7]:
polity_rows = comparison[comparison["has_treccani_polity"] & comparison["cultura_sub_polities"].notna()]
window_rows = comparison[comparison["has_treccani_window"] & comparison["cultura_sub_polities"].notna()]
pd.DataFrame({
    "field": ["polity", "productivity window"],
    "correct": [polity_rows["polity_ok"].sum(), window_rows["window_ok"].sum()],
    "individuals": [len(polity_rows), len(window_rows)],
    "accuracy": [polity_rows["polity_ok"].mean().round(2), window_rows["window_ok"].mean().round(2)],
})

,field,correct,individuals,accuracy
0,polity,131,169,0.78
1,productivity window,155,176,0.88
